## Setup

In [1]:
#FPGA notebook
from tqdm.notebook import tqdm
from qick import *
#%pylab inline
#import h5py
#import json
#from scipy.signal import find_peaks
import zmq
import numpy as np
import threading

In [2]:
soc = QickSoc()
soccfg = soc
print(soccfg)


QICK configuration:

	Board: RFSoC4x2

	Software version: 0.2.267
	Firmware timestamp: Wed Sep  6 18:49:29 2023

	Global clocks (MHz): tProcessor 409.600, RF reference 491.520

	2 signal generator channels:
	0:	axis_signal_gen_v6 - envelope memory 65536 samples (6.667 us)
		fs=9830.400 MHz, fabric=614.400 MHz, 32-bit DDS, range=9830.400 MHz
		DAC tile 0, blk 0 is DAC_B
	1:	axis_signal_gen_v6 - envelope memory 65536 samples (6.667 us)
		fs=9830.400 MHz, fabric=614.400 MHz, 32-bit DDS, range=9830.400 MHz
		DAC tile 2, blk 0 is DAC_A

	2 readout channels:
	0:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 MHz, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		maxlen 16384 accumulated, 1024 decimated (1.852 us)
		triggered by output 7, pin 14, feedback to tProc input 0
		ADC tile 0, blk 0 is ADC_D
	1:	axis_readout_v2 - configured by PYNQ
		fs=4423.680 MHz, decimated=552.960 MHz, 32-bit DDS, range=4423.680 MHz
		maxlen 16384 accumulated, 1024 decimated (1.852 us)
		triggered by o

### Hardware Configuration

You should cable the board in loopback, connecting one DAC to one ADC. You can set appropriate generator and readout numbers in the cell below, which will be used in `res_ch` and `ro_chs` entries in the program dictionaries below.

- start with nothing

In [3]:
class LoopbackProgram(AveragerProgram): #inherit from AveragerProgram
    def initialize(self):
        cfg=self.cfg #get config
        res_ch = cfg["res_ch"] #DAC channel select
        # configure DAC - set the nyquist zone
        self.declare_gen(ch=cfg["res_ch"], nqz=1)        

        # configure ADC - set the readout lengths and downconversion frequencies (ensuring it is an available DAC frequency)
        # 
        for ch in cfg["ro_chs"]:        
            self.declare_readout(
                ch=ch, 
                length=self.cfg["readout_length"],
                freq=self.cfg["pulse_freq"], 
                gen_ch=cfg["res_ch"]
            )

        # convert frequency to DAC frequency (ensuring it is an available ADC frequency)
        freq = self.freq2reg(cfg["pulse_freq"],gen_ch=res_ch, ro_ch=cfg["ro_chs"][0])
        phase = self.deg2reg(cfg["res_phase"], gen_ch=res_ch)
        gain = cfg["pulse_gain"]
        self.default_pulse_registers(ch=res_ch, freq=freq, phase=phase, gain=gain)

        # configure pulse style (ie. square, sinc, etc.)
        style=self.cfg["pulse_style"]

        if style in ["flat_top","arb"]:
            sigma = cfg["sigma"]
            self.add_gauss(ch=res_ch, name="measure", sigma=sigma, length=sigma*5)
        if style == "const":
            self.set_pulse_registers(ch=res_ch, style=style, length=cfg["length"])
        elif style == "flat_top":
            # The first half of the waveform ramps up the pulse, the second half ramps down the pulse
            self.set_pulse_registers(ch=res_ch, style=style, waveform="measure", length=cfg["length"])
        elif style == "arb":
            self.set_pulse_registers(ch=res_ch, style=style, waveform="measure")
        
        self.synci(200)  # give processor some time to configure pulses [clock ticks]
    
    def body(self):
        # fire the pulse
        # trigger all declared ADCs
        # pulse PMOD0_0 for a scope trigger
        # pause the tProc until readout is done
        # increment the time counter to give some time before the next measurement
        # (the syncdelay also lets the tProc get back ahead of the clock)
        
        # self.measure(pulse_ch=self.cfg["res_ch"], 
        #              adcs=self.ro_chs,
        #              pins=[0], 
        #              adc_trig_offset=self.cfg["adc_trig_offset"],
        #              wait=True,
        #              syncdelay=self.us2cycles(self.cfg["relax_delay"]))
        
        # equivalent to the following:
        self.trigger(adcs=self.ro_chs,
                     pins=[0], 
                     adc_trig_offset=self.cfg["adc_trig_offset"])
        self.pulse(ch=self.cfg["res_ch"])
        self.wait_all()
        self.sync_all(self.us2cycles(self.cfg["relax_delay"]))

In [5]:
class FPGAController:

    def __init__(self):
        self.config = None
        self.frequency = None

    def configure(self, config):
        self.config = config
        print("FPGA configured:")
        print(config)

    def set_frequency(self, frequency):
        if self.config is None:
            raise RuntimeError("FPGA has not been configured")
        self.frequency = frequency
        self.config["pulse_freq"] = self.frequency
        print(f"Frequency set to {frequency} Hz")

    def measure(self):
        if self.config is None:
            raise RuntimeError("FPGA has not been configured")
        if self.frequency is None:
            raise RuntimeError("Frequency has not been set")
        
        
        prog = LoopbackProgram(soccfg, self.config)
        adc = prog.acquire_decimated(soc, progress=False)

        I = adc[:, 0]
        Q = adc[:, 1]

        return I.astype(np.float32) + 1j * Q.astype(np.float32)

    def status(self):
        return {
            "connected": True,
            "frequency": self.frequency,
            "configured": self.config is not None
        }


class FPGAServer:

    def __init__(self, fpga, port=5555):
        self.fpga = fpga
        self.running = threading.Event()
        self.running.set()

        self.context = zmq.Context()
        self.socket = self.context.socket(zmq.REP)

        try:
            self.socket.bind(f"tcp://0.0.0.0:{port}")
        except zmq.ZMQError:
            self.socket.close(0)
            self.context.term()
            raise

        self.poller = zmq.Poller()
        self.poller.register(self.socket, zmq.POLLIN)

        print(f"FPGA server listening on port {port}")

    def run(self):
        try:
            while self.running.is_set():

                events = dict(self.poller.poll(100))

                if self.socket not in events:
                    continue

                try:
                    request = self.socket.recv_json()
                    command = request.get("command")
                    parameters = request.get("parameters", {})

                    print(f"Received command: {command}")

                    if command == "configure":
                        self.fpga.configure(parameters["config"])
                        self.socket.send_json({"status": "ok"})

                    elif command == "set_frequency":
                        self.fpga.set_frequency(parameters["frequency"])
                        self.socket.send_json({"status": "ok"})

                    elif command == "measure":
                        iq = self.fpga.measure()
                        self.socket.send(
                            iq.astype(np.complex64).tobytes()
                        )

                    elif command == "status":
                        self.socket.send_json({
                            "status": "ok",
                            "data": self.fpga.status()
                        })

                    elif command == "stop":
                        print("Shutdown command received")
                        self.socket.send_json({"status": "ok"})
                        self.running.clear()

                    else:
                        self.socket.send_json({
                            "status": "error",
                            "error": f"Unknown command: {command}"
                        })

                except Exception as e:
                    print(f"Error handling request: {e}")

                    try:
                        self.socket.send_json({
                            "status": "error",
                            "error": str(e)
                        })
                    except zmq.ZMQError:
                        pass

        except KeyboardInterrupt:
            print("\nKeyboard interrupt received")

        finally:
            self.stop()

    def stop(self):
        if self.running.is_set():
            print("Stopping FPGA server...")
            self.running.clear()

        try:
            self.poller.unregister(self.socket)
        except Exception:
            pass

        try:
            self.socket.close(0)
        except Exception:
            pass

        try:
            self.context.term()
        except Exception:
            pass

        print("FPGA server stopped")


if __name__ == "__main__":
    fpga = FPGAController()
    server = FPGAServer(fpga, port=5555)

    try:
        server.run()
    finally:
        server.stop()



FPGA server listening on port 5555
Received command: status
Received command: configure
FPGA configured:
{'res_ch': 1, 'ro_chs': [1], 'reps': 1, 'relax_delay': 1.0, 'res_phase': 0, 'pulse_style': 'const', 'sigma': 50, 'length': 300, 'readout_length': 100, 'pulse_gain': 40000, 'adc_trig_offset': 250, 'soft_avgs': 200}
Received command: set_frequency
Frequency set to 5000 Hz
Received command: measure
Received command: status
Received command: configure
FPGA configured:
{'res_ch': 1, 'ro_chs': [1], 'reps': 1, 'relax_delay': 1.0, 'res_phase': 0, 'pulse_style': 'const', 'sigma': 50, 'length': 300, 'readout_length': 100, 'pulse_gain': 40000, 'adc_trig_offset': 250, 'soft_avgs': 200}
Received command: set_frequency
Frequency set to 5000 Hz
Received command: measure
Received command: status
Received command: configure
FPGA configured:
{'res_ch': 1, 'ro_chs': [1], 'reps': 1, 'relax_delay': 1.0, 'res_phase': 0, 'pulse_style': 'const', 'sigma': 50, 'length': 300, 'readout_length': 100, 'pulse_gai

## Alban demo

In [ ]:
#GEN_CH = 0-1 for DAC_B-DAC_A 
#RO_CH = 0-4 for ADC_D-ADC_A

GEN_CH = 1 # DAC A
RO_CH = 1  # ADC C

# GEN_CH_B = 0 # DAC B
# RO_CH_D = 0  # ADC D

config={"res_ch":GEN_CH, # --Fixed
        "ro_chs":[RO_CH], # --Fixed
        "reps":1, # --Fixed
        "relax_delay":1.0, # --us
        "res_phase":0, # --degrees
        "pulse_style": "const", # --Fixed

        # Try varying  length from 10-100 clock ticks
        "sigma": 50, # [Clock ticks]
        # Try varying sigma from 10-50 clock ticks
        
        "length":300, # [Clock ticks]
        # Try varying length from 10-100 clock ticks
        
        "readout_length":100, # [Clock ticks]
        # Try varying readout_length from 50-1000 clock ticks

        "pulse_gain":40000, # [DAC units]
        # Try varying pulse_gain from 500 to 30000 DAC units

        "adc_trig_offset": 250, # [Clock ticks]
        # Try varying adc_trig_offset from 100 to 220 clock ticks
        # I think this is like how many ticks until the ADC starts listening

        "soft_avgs":200
        # Try varying soft_avgs from 1 to 200 averages

       }


i=[]
q=[]

#Declare no. of points and start/end freqs
steps = 1001
expt_cfg = {
    "start": 6286,
    "end": 6536,
}

fpts = np.linspace(expt_cfg["start"], expt_cfg["end"], steps)

#Adjust freq in'config' and measure  
for f in tqdm(fpts): 

    config["pulse_freq"] = f
    
    prog =LoopbackProgram(soccfg, config) 
    adc1 = prog.acquire_decimated(soc, progress=False)
    
    i.append(np.abs(adc1[0][0] + 1j * adc1[0][1]))
    
i = np.array(i)

S21 = np.abs(i)
S21 = np.array(S21)

time_axis = soccfg.cycles2us(np.arange(config["readout_length"]), gen_ch=GEN_CH)
update_global_max(np.max(S21))

In [ ]:
log_S21_data = np.where(S21 != 0, 20 * np.log10(S21 / global_max), -40.0)
# log_S21_data = np.log10(S21/np.max(S21))

plt.figure(figsize=(12, 4))  
plt.subplot(1, 2, 1)
plt.pcolormesh(np.arange(config["readout_length"]), fpts, S21, cmap='viridis', shading='auto')
# Labels and title
plt.xlabel("Clock Ticks")
plt.ylabel("Iteration (Inverse)")
plt.colorbar(label="Magnitude (adc levels)")
# plt.xlim(0,80)



average_vertical_slice = np.mean(log_S21_data, axis=1)
vertical_slice = log_S21_data[:, 5]

# Mask to region of interest
mask = (fpts >= 6400) & (fpts <= 6450)
fpts_roi = fpts[mask]
data_roi = average_vertical_slice[mask]

peaks, _ = find_peaks(data_roi, prominence=0.25)
dips, _ = find_peaks(-data_roi, prominence=0.25)

peak_freqs = fpts_roi[peaks]
dip_freqs = fpts_roi[dips]

print("Peaks at:", peak_freqs)
print("Dips at:", dip_freqs)

plt.subplot(1, 2, 2)
plt.plot(fpts, average_vertical_slice, marker='o')
# plt.plot(fpts, vertical_slice, marker='o')
plt.xlabel("Frequency (MHz)")
plt.ylabel("dB")
plt.grid(True)
min_freq = fpts[np.argmin(average_vertical_slice)]
plt.axvline(min_freq, color='r', linestyle='--', label=f'Min at {min_freq:.2f} MHz')
max_freq = fpts[np.argmax(average_vertical_slice)]
plt.axvline(max_freq, color='r', linestyle='--', label=f'Max at {max_freq:.2f} MHz')

plt.plot(fpts_roi, data_roi)
plt.plot(fpts_roi[peaks], data_roi[peaks], 'ro', label='Peaks')
plt.plot(fpts_roi[dips], data_roi[dips], 'bo', label='Dips')
plt.legend()

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fr=min_freq
# fr=max_freq
# fr=fpts_roi[peaks][-1]
# fr=4997
print(fr)
# print(fpts_total)

## FMR Sweep using big magnet

5/10/26: FPGA 'pulse style = const' generates an AC signal of frequency 'pulse freq'. Previously thought 'const' generates a square wave. Thus must effectively excite the YIG sample for all freqs across magnetic field strength (FMR). 

So FPGA experimentation will consist of a VNA style frequency sweep coupled with the magnet for resonance frequencies for all magnetic field strengths tested ('fmr sweep') followed by a sweep through the same magnetic field stengths with a short time-domain pulse with the pulse freq set to the fmr frequency for that corresponding magnetic field stength ('pulse sweep'). 

For 'FMR Sweep';   start flask server -> receive freqs + currents from PC           -> measure + send data to PC -> stop server

For 'Pulse Sweep'; start flask server -> receive resonance freqs + currents from PC -> measure + send data to PC -> stop server

The PC notebook should take on the bulk of the workload as opposed to the FPGA. FPGA should be treated similar to VNA in the sense that all configuration and setup is done on th PC end. FPGA should simply take config file from PC and send recorded data back to PC. This reduces a lot of overhead caused by constant parity checks and completely eliminates any issues caused by it.